# 01 — Data Understanding

**Phase 2** of the project plan: load the raw freMTPL2 tables exactly as downloaded, inspect their structure, and reconcile the frequency table against the severity table.

This notebook does not clean or join the data for modelling — it only observes and documents. All findings below are also written up in `reports/data_dictionary.md`. Cleaning decisions (capping, exclusions, encoding fixes) are deferred to Phase 3, so that every correction is a deliberate, recorded choice rather than something silently baked into a loading function.

All logic used here lives in `src/auto_pricing/data.py` and `src/auto_pricing/validation.py`, and is unit-tested in `tests/test_validation.py` — this notebook only calls that code and narrates what it finds.

In [1]:
import pandas as pd

from auto_pricing.data import load_frequency, load_severity, aggregate_severity_by_policy
from auto_pricing.validation import (
    missing_value_report,
    duplicate_id_count,
    numeric_range_report,
    find_orphan_claims,
    reconcile_claim_counts,
)

pd.set_option("display.max_columns", None)

freq = load_frequency()
sev = load_severity()

print("frequency table:", freq.shape)
print("severity table: ", sev.shape)

frequency table: (678013, 12)
severity table:  (26639, 2)


## 1. Schema

`freq` should be one row per policy (`IDpol` unique). `sev` is one row per **claim**, so `IDpol` repeating there is expected, not an error — it just means that policy had more than one claim.

In [2]:
freq.dtypes

IDpol           int64
ClaimNb         int64
Exposure      float64
Area              str
VehPower        int64
VehAge          int64
DrivAge         int64
BonusMalus      int64
VehBrand          str
VehGas            str
Density         int64
Region            str
dtype: object

In [3]:
freq.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region
0,1,1,0.10,D,5,0,55,50,B12,'Regular',1217,R82
1,3,1,0.77,D,5,0,55,50,B12,'Regular',1217,R82
2,5,1,0.75,B,6,2,52,50,B12,'Diesel',54,R22
3,10,1,0.09,B,7,0,46,50,B12,'Diesel',76,R72
4,11,1,0.84,B,7,0,46,50,B12,'Diesel',76,R72


In [4]:
sev.head()

,IDpol,ClaimAmount
0,1552,995.20
1,1010996,1128.12
2,4024277,1851.11
3,4007252,1204.00
4,4046424,1204.00


## 2. Missingness and duplicate keys

Both tables turn out to be complete (no nulls), and `IDpol` is a genuine primary key in `freq`.

In [5]:
print("missing values, freq:")
print(missing_value_report(freq))
print()
print("missing values, sev:")
print(missing_value_report(sev))
print()
print("duplicate IDpol rows in freq:", duplicate_id_count(freq, "IDpol"))

missing values, freq:
IDpol         0
ClaimNb       0
Exposure      0
Area          0
VehPower      0
VehAge        0
DrivAge       0
BonusMalus    0
VehBrand      0
VehGas        0
Density       0
Region        0
dtype: int64

missing values, sev:
IDpol          0
ClaimAmount    0
dtype: int64

duplicate IDpol rows in freq: 0


## 3. Numeric ranges

Looking for impossible values, not just "interesting" ones. Two things stand out immediately:

- `Exposure` should be a fraction of a year (0 to 1), but its max is above 1.
- `ClaimNb` has a max of 16 on a single policy, which combined with the exposure figures below turns out to be implausible.

Both are recorded as anomalies in `reports/data_dictionary.md` rather than fixed here.

In [6]:
numeric_range_report(
    freq, ["ClaimNb", "Exposure", "VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]
)

,min,max,mean,n_negative
column,,,,
ClaimNb,0.000000,16.00,0.053247,0
Exposure,0.002732,2.01,0.528750,0
VehPower,4.000000,15.00,6.454631,0
VehAge,0.000000,100.00,7.044265,0
DrivAge,18.000000,100.00,45.499122,0
BonusMalus,50.000000,230.00,59.761502,0
Density,1.000000,27000.00,1792.422405,0


In [7]:
n_over_exposed = (freq["Exposure"] > 1).sum()
print(f"policies with Exposure > 1: {n_over_exposed} ({n_over_exposed / len(freq):.3%} of policies)")

freq[freq["ClaimNb"] >= 8][["IDpol", "ClaimNb", "Exposure", "DrivAge", "BonusMalus"]]

policies with Exposure > 1: 1224 (0.181% of policies)


,IDpol,ClaimNb,Exposure,DrivAge,BonusMalus
6539,19471,11,1.00,65,50
320764,2239279,8,0.41,52,50
321248,2241683,16,0.33,52,50
321512,2248174,9,0.08,52,50
487268,3253234,11,0.08,53,50
488270,3254353,11,0.07,53,50


16 claims in 4 months (`IDpol=2241683`, `Exposure=0.33`) is not a credible number of at-fault liability claims for one private vehicle. scikit-learn's own tutorial on this dataset caps `ClaimNb` for exactly this reason — we will make and record the same kind of decision explicitly in Phase 3, rather than let a handful of extreme rows dominate a Poisson GLM's likelihood.

## 4. Categorical levels

Checking what values each categorical column actually takes, since a stray or malformed category is easy to miss until it silently becomes its own dummy variable.

In [8]:
for col in ["Area", "VehBrand", "VehGas", "Region"]:
    print(f"{col} ({freq[col].nunique()} levels):", sorted(freq[col].unique()))

Area (6 levels): ['A', 'B', 'C', 'D', 'E', 'F']
VehBrand (11 levels): ['B1', 'B10', 'B11', 'B12', 'B13', 'B14', 'B2', 'B3', 'B4', 'B5', 'B6']


VehGas (2 levels): ["'Diesel'", "'Regular'"]
Region (22 levels): ['R11', 'R21', 'R22', 'R23', 'R24', 'R25', 'R26', 'R31', 'R41', 'R42', 'R43', 'R52', 'R53', 'R54', 'R72', 'R73', 'R74', 'R82', 'R83', 'R91', 'R93', 'R94']


`VehGas` shows `'Diesel'` and `'Regular'` *with the quote characters embedded in the string itself* — an artifact of how the source ARFF file was parsed, not a real third fuel type. Harmless for now, but it must be stripped before one-hot encoding in Phase 4, or the model would silently treat it as a legitimate category label.

## 5. Reconciling frequency against severity

This is the central integrity check for the whole project: does the claim count each policy *reports* (`ClaimNb`) match how many claim records actually *exist* for it in the severity table? If not, which number should the models trust?

In [9]:
orphans = find_orphan_claims(freq, sev)
print(f"severity rows with no matching policy: {len(orphans)}")
print(f"...belonging to {orphans['IDpol'].nunique()} distinct policies")

orphans.groupby("IDpol").agg(n_claims=("ClaimAmount", "size"), total_amount=("ClaimAmount", "sum"))

severity rows with no matching policy: 195
...belonging to 6 distinct policies


,n_claims,total_amount
IDpol,,
2220367,24,84441.63
2227533,25,64093.76
2262511,66,151800.59
2277846,23,73440.72
2282134,36,330920.05
2286775,21,84017.43


Only 6 *policies* are orphaned, but they are unusually claim-heavy — together they account for 195 claim rows and roughly €789k in claim value. A quick manual check that only compares unique IDs would report "6 rows missing" and understate how much claim value is actually affected; this is why the check is written as a real, tested function rather than eyeballed.

In [10]:
mismatches = reconcile_claim_counts(freq, sev)
print(f"policies where ClaimNb disagrees with actual severity row count: {len(mismatches)}")
mismatches.head(10)

policies where ClaimNb disagrees with actual severity row count: 9123


,IDpol,ClaimNb,ClaimNbFromSev
0,1,1.0,0.0
1,3,1.0,0.0
2,5,1.0,0.0
3,10,1.0,0.0
4,11,1.0,0.0
5,13,1.0,0.0
6,15,1.0,0.0
7,17,1.0,0.0
8,18,1.0,0.0
9,21,1.0,0.0


In [11]:
print("total ClaimNb across all policies:", freq["ClaimNb"].sum())
print("total rows in severity table:      ", len(sev))
print("total claim amount in severity:     \u20ac{:,.2f}".format(sev["ClaimAmount"].sum()))

total ClaimNb across all policies: 36102
total rows in severity table:       26639
total claim amount in severity:     €60,697,930.68


`ClaimNb` sums to 36,102 across all policies, but there are only 26,639 rows in the severity table. The two tables are describing claim activity from somewhat different bookkeeping, and neither is simply "wrong" by construction — Phase 3 has to pick a policy for which source drives the frequency target.

## 6. Severity distribution shape

A quick look at how skewed `ClaimAmount` is, since that shape is exactly why Phase 6 models it with a Gamma (not a Normal/OLS) distribution.

In [12]:
sev["ClaimAmount"].describe()

count    2.663900e+04
mean     2.278536e+03
std      2.929748e+04
min      1.000000e+00
25%      6.868100e+02
50%      1.172000e+03
75%      1.228080e+03
max      4.075401e+06
Name: ClaimAmount, dtype: float64

In [13]:
print("largest 10 claims:")
sev.nlargest(10, "ClaimAmount")

largest 10 claims:


,IDpol,ClaimAmount
9756,1120377,4075400.56
15768,110846,1403057.40
11637,2141337,1301172.60
17992,3122016,774411.50
2869,2008127,390742.27
3307,3025890,369131.88
9999,1117644,307096.42
5521,158309,301635.49
18627,3075820,287423.00
18295,3150210,281897.49


The largest claim (€4,075,400.56) is roughly 3,500× the median (€1,172). This kind of tail is normal for motor third-party liability, where a small number of severe bodily-injury claims can be very expensive — but it means a handful of records can dominate an unweighted mean-squared-error metric, which is part of why Phase 6 and Phase 7 use exposure/claim-weighted deviance instead of plain RMSE as the primary metric.

## 7. Summary of anomalies found

Full detail and reasoning for each is in `reports/data_dictionary.md`. In brief:

1. 1,224 policies have `Exposure > 1` (max 2.01) — exposure should not exceed one year.
2. 9,123 policies have `ClaimNb` that disagrees with their actual severity row count, including a few with implausibly high claim counts relative to their exposure.
3. 6 policies referenced in the severity table are entirely absent from the frequency table, accounting for 195 claim rows (~€789k).
4. `ClaimAmount` has a long right tail (max ~3,500× the median) — expected for liability claims, but needs explicit large-loss handling later.
5. `VehGas` category labels contain literal embedded quote characters.
6. `VehAge` has a maximum of 100 years, which is not a plausible vehicle age.

**Nothing has been dropped, capped, or corrected in this notebook.** These become explicit, written cleaning decisions in Phase 3 (`notebooks/02_data_cleaning_eda.ipynb`), which is also where the two tables get joined into a single policy-level modelling table.